# 08 – Forecasts and fit: PINN vs Mechanistic

Comparison of the probabilistic forecasts of the two models (optionally also of a baseline). One notebook for the four synthetic scenarios and for the real case, chosen with `CASE` (as in notebooks 03–07):

| Figure | `"synthetic"` (R0 = 1.5 / 2.0, noise `"light"` / `"mild"`) | `"real"` |
|---|---|---|
| **Forecast grid**: one panel per rolling window, with data, 90% and 50% bands and median | yes | yes |
| **Fixed horizons**: for every week the forecast made 1, 2, 3, 4 weeks before | yes | yes |
| **Retrospective fit**: fit of the window of the last week T_LAST, PINN vs Mechanistic (+ coverage of the 90% interval) | yes | – |
| **Fit by age class**: sanity check, true incidence of every age class vs the fit of the two models | yes | – |

The x axis shows the week number (synthetic scenarios) or the date (real case). Observations: new infections (synthetic) or new deaths (real).

**T_LAST** is the last week of the plots of the synthetic scenarios and depends on the scenario: `t0 = 33` (week 34) for R0 = 1.5 and `t0 = 23` (week 24) for R0 = 2.0 (`case.t0_last`). The windows whose forecast ends after T_LAST are not shown. The real case goes from 2020-03-15 to 2020-07-19.

Input (from `precomputed/` or `outputs/`, `SOURCE`):
- `posteriors/pinn/<case>/<run>/forecast_h<H>/forecast_results.pickle` (notebook 04; `<run>` is `best_WIS` or `seed_<seed>`, see `PINN_SEED`). For the retrospective fit and the fit by age class notebook 04 must have been run on the synthetic scenario with the window T_LAST (it is part of `case.t0_estimation()`) and, for the age classes, with `AGE_FORECASTS = True`.
- `posteriors/mechanistic/<case>/forecast_h<H>/forecast_results.pickle` and `abc_distributions.pkl` (notebook 05). The retrospective fit and the fit by age class use `last_week` of `abc_distributions.pkl` (trajectories of the folder `retro_w34` / `retro_w24`).
- Optional: `baselines/<case>/forecast_results_baseline.pkl` (notebook 06).

Output: `figures/forecasts/<case>/`.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np

from pinnvsmech.data.synthetic import load_age_incidence
from pinnvsmech.evaluation.best_seed import resolve_run_dir
from pinnvsmech.forecasting.baseline import baseline_label
from pinnvsmech.forecasting.windows import select_windows
from pinnvsmech.inference.cases import load_case
from pinnvsmech.paths import FIGURES_DIR, OUTPUTS_DIR, PRECOMPUTED_DIR
from pinnvsmech.plotting.forecasts import (
    describe_forecasts, fit_coverage, plot_age_forecast_compare, plot_fixed_horizon_grid,
    plot_probabilistic_forecasts, plot_single_window,
)
from pinnvsmech.plotting.params import load_distributions

## Settings

In [ ]:
CASE = "synthetic"                 # "synthetic" or "real"
R0_TRUE = 1.5                      # synthetic only: 1.5 or 2.0
NOISE = "light"                    # synthetic only: "light" or "mild"
CASE_OVERRIDES = dict(dt_train=None)   # look-back on all the history, as the Mechanistic "nall" results (it is
                                   # the default of the synthetic scenarios; for the real case the default is 5 weeks)

SOURCE = "precomputed"             # "precomputed" (results of the paper) or "outputs" (new runs)
PINN_SEED = "best"                 # PINN run: "best" = folder best_WIS of the case, or the seed chosen in notebook 10; or the seed number (e.g. 10) to use seed_010
HORIZON = 4                        # horizon of the forecast files (weeks)
HORIZONS_SHOWN = (1, 2, 3, 4)      # panels of the figure with fixed horizons

SHOW_BASELINE = False              # add the baseline to the figures (needs the file of notebook 06)

FIGURES = ["grid", "horizons", "retrospective", "age"]   # figures to make; the last two only for the synthetic scenarios
SAVE_FIGURES = False
FIGURE_FORMAT = "pdf"              # "pdf" or "png"

Plot settings of every case (y limits, ticks, layout). They are the ones of the paper; edit them to change the figures.

- Synthetic scenarios: the observations and the bands stop at T_LAST. `drop_last` is the number of windows removed at the end of the grid. For the figure of the horizons, R0 = 1.5 is the left figure (with the labels of the axes, no legend) and R0 = 2.0 the right one (with the legend, no labels) of the side-by-side layout of the paper.
- Real case: `ylim` of the grid is one limit for every row (the first windows have wider bands); `ytick_step` is one step for every horizon.

In [ ]:
M_LEFT = dict(left=0.20, right=0.98, bottom=0.09, top=0.98, hspace=0.30)   # with the labels of the axes
M_RIGHT = dict(M_LEFT, left=0.07)                                           # without the labels

SYNTHETIC_PLOT = {
    1.5: dict(grid_ylim=(-30000, 700000), drop_last=6,
              horizons=dict(figsize=(9, 13), margins=M_LEFT, ytick_step=1e5, show_legend=False,
                            show_ylabel=True, show_side_label=True),
              window=dict(ylim=(-50000, 620000), ytick_step=1e5)),
    2.0: dict(grid_ylim=(-150000, 2150000), drop_last=1,
              horizons=dict(figsize=(7.71, 13), margins=M_RIGHT, ytick_step=3e5, show_legend=True,
                            show_ylabel=False, show_side_label=False),
              window=dict(ylim=(-150000, 1600000), ytick_step=3e5)),
}
REAL_PLOT = dict(
    grid_ylim=[(-300, 12000), (-200, 4000), (-100, 1500), (-100, 1500)], drop_last=3,
    horizons=dict(figsize=(9, 13), ytick_step=[500, 1000, 2000, 2000], legend_fontsize=13,
                  margins=dict(left=0.20, right=0.98, bottom=0.09, top=0.98, hspace=0.43)),
)
WINDOW_LEGEND = None               # retrospective fit: None (no legend), "figure" (above the panel) or "axes"

## Data and forecasts

In [ ]:
case = load_case(CASE, R0=R0_TRUE, noise=NOISE, **CASE_OVERRIDES)
# The Mechanistic results do not depend on the weights of the likelihood
mech_label = load_case(CASE, R0=R0_TRUE, noise=NOISE,
                       **{k: v for k, v in CASE_OVERRIDES.items() if k != "weight"}).label

root = PRECOMPUTED_DIR if SOURCE == "precomputed" else OUTPUTS_DIR
pinn_dir = resolve_run_dir(PINN_SEED, root / "posteriors" / "pinn" / case.label)
mech_dir = root / "posteriors" / "mechanistic" / mech_label
baseline_dir = root / "baselines" / baseline_label(case)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


forecasts = {
    "pinn": load_pickle(pinn_dir / f"forecast_h{HORIZON}" / "forecast_results.pickle"),
    "mech": load_pickle(mech_dir / f"forecast_h{HORIZON}" / "forecast_results.pickle"),
}
if SHOW_BASELINE:
    forecasts["baseline"] = load_pickle(baseline_dir / "forecast_results_baseline.pkl")

print(case.label, "| PINN run", pinn_dir.name)
describe_forecasts(forecasts)

## Last week of the plots and layout of every case

Synthetic: `T_LAST` = 33 (R0 = 1.5) or 23 (R0 = 2.0); data and forecasts are cut there, and the windows with `t0 + HORIZON > T_LAST` are removed. Real: from the week of 2020-03-15 to the week of 2020-07-19.

In [ ]:
fc, t0_min, t_last = select_windows(case, forecasts, HORIZON)     # trimmed for the synthetic scenarios

if CASE == "synthetic":
    T_LAST = case.t0_last                                  # 33 (R0 = 1.5) / 23 (R0 = 2.0)
    plot = SYNTHETIC_PLOT[float(R0_TRUE)]
    obs = case.obs[:T_LAST + 1]
    dates = case.dates[:T_LAST + 1]
    grid_kw = dict(
        grid_size="all", ylim=plot["grid_ylim"], drop_last=plot["drop_last"], show_fit=False,
        ylabel="Weekly new infections",
        title=f"Weekly new infections Probabilistic Forecast - Synthetic Scenario: $R_0 = {R0_TRUE}$",
    )
    horizon_kw = dict(plot["horizons"], ylabel="New infections")
    t0_min, t_last = None, None                           # the cut is already done
    print(f"T_LAST = {T_LAST} (week {T_LAST + 1}) | observations: {len(obs)} weeks")
else:
    T_LAST = None
    plot = REAL_PLOT
    obs, dates = case.obs, case.dates
    grid_kw = dict(
        grid_size=4, ylim=plot["grid_ylim"], drop_last=plot["drop_last"], show_fit=False, t0_min=t0_min,
        t_last=t_last, ylabel="Weekly Deaths",
        title="COVID-19 Weekly Deaths Probabilistic Forecast - London 2020",
    )
    horizon_kw = dict(plot["horizons"], ylabel="Weekly Deaths")
    print(f"weeks shown: t0 = {t0_min} (2020-03-15) to {t_last} (2020-07-19)")

describe_forecasts(fc)

fig_dir = FIGURES_DIR / "forecasts" / case.label
os.makedirs(fig_dir, exist_ok=True)


def save(fig, name, **kwargs):
    if SAVE_FIGURES:
        fig.savefig(fig_dir / f"{name}.{FIGURE_FORMAT}", dpi=300, **kwargs)

## Forecast grid

One panel per window: the observations (open circles: all the data; filled: look-back window of the window), and the 90% / 50% bands and the median of every model, from the current week on.

In [ ]:
if "grid" in FIGURES:
    fig = plot_probabilistic_forecasts(
        dates=dates, obs=obs, dt_train=case.dt_train,
        forecasts_pinn=fc["pinn"], forecasts_mech=fc["mech"],
        forecasts_baseline=fc.get("baseline"),
        **grid_kw,
    )
    save(fig, f"forecast_grid_h{HORIZON}", bbox_inches="tight")
    plt.show()

## Forecasts at fixed horizon

One panel per horizon: for every week, the forecast made `h` weeks earlier (box: 90% and 50% intervals, median).

In [ ]:
if "horizons" in FIGURES:
    fig, _ = plot_fixed_horizon_grid(
        dates, obs, horizons=HORIZONS_SHOWN,
        forecasts_pinn=fc["pinn"], forecasts_mech=fc["mech"], forecasts_baseline=fc.get("baseline"),
        t0_min=t0_min, t_last=t_last, style="box", **horizon_kw,
    )
    save(fig, f"forecast_fixed_horizons_h{HORIZON}")
    plt.show()

## Retrospective fit (synthetic scenarios)

Fit of the window of the last week T_LAST (all the history up to that week): PINN (window `t0 = T_LAST` of the forecast file, not trimmed) and Mechanistic model (simulated trajectories of `retro_w<T_LAST+1>`), with the true observations. `fit_coverage` prints the fraction of observations inside the 90% interval of each model.

In [ ]:
if CASE == "synthetic" and ("retrospective" in FIGURES or "age" in FIGURES):
    data_mech = load_distributions(mech_dir / "abc_distributions.pkl")
    if T_LAST not in data_mech.get("last_week", {}):
        raise KeyError(f"No extra fit at t0 = {T_LAST} in the Mechanistic results: convert the folder "
                       f"'retro_w{T_LAST + 1}' with notebook 05")
    last_week = data_mech["last_week"][T_LAST]
    print("Mechanistic extra fit at t0 =", T_LAST, "| keys:", sorted(last_week))

In [ ]:
if CASE == "synthetic" and "retrospective" in FIGURES:
    mech_traj = last_week["trajectories"]                     # (n_samples, n_weeks)
    window_kw = dict(plot["window"], figsize=(10, 4), tick_fontsize=17, legend_fontsize=14.5, xtick_step=2)
    if WINDOW_LEGEND is None:
        window_kw.update(legend=None, margins=dict(left=0.11, right=0.98, bottom=0.22, top=0.93))
    else:
        window_kw.update(legend=WINDOW_LEGEND)

    fig, ax = plot_single_window(
        case.dates, case.obs, t0=T_LAST,
        forecasts_pinn=forecasts["pinn"],                     # not trimmed: it has the window T_LAST
        mech_samples=mech_traj, **window_kw,
    )
    save(fig, f"fit_retrospective_week{T_LAST + 1}")
    plt.show()

    cov_pinn, cov_mech, n = fit_coverage(case.obs, T_LAST, forecasts["pinn"], mech_traj)
    print(f"{case.label}: coverage of the 90% interval, PINN {100 * cov_pinn:.0f}% | "
          f"Mechanistic {100 * cov_mech:.0f}%  ({n} weeks)")

## Fit by age class (synthetic scenarios, sanity check)

True incidence of every age class (no noise, dashed) against the fit of the window T_LAST of the two models: left the PINN (forecast of every age class, notebook 04 with `AGE_FORECASTS = True`), right the Mechanistic model (`infections_by_age.npz` of `retro_w<T_LAST+1>`). `AGE_ROWS` are the age classes of every row, numbered from 1.

In [ ]:
AGE_ROWS = ((1, 2, 3, 4, 5), (7, 9, 11, 13, 16))     # e.g. also ((1, 2, 4), (7, 11, 13, 16))

if CASE == "synthetic" and "age" in FIGURES:
    _, age_incidence = load_age_incidence(R0_TRUE)
    true_age = np.asarray(age_incidence)[:T_LAST + 1]         # (T_LAST + 1, 16), no noise
    forecasts_age = {a: load_pickle(pinn_dir / f"forecast_age{a}" / "forecast_results.pickle")
                     for a in range(true_age.shape[1])}

    fig = plot_age_forecast_compare(
        forecasts_age=forecasts_age, mech_samples_age=last_week["weekly_age"], dates=dates, true_age=true_age,
        t0=T_LAST, age_rows=AGE_ROWS,
        title=f"Weekly incidence by age class - synthetic scenario ($R_0 = {R0_TRUE}$)",
    )
    save(fig, f"fit_by_age_week{T_LAST + 1}", bbox_inches="tight")
    plt.show()